# 07 - GitHub Actions

This replaces SageMaker Projects, Service Catalog, CodePipeline and CodeBuild.

### Why

The workshop's model-build CI/CD needed: a Service Catalog product, a CodeStar
connection tagged `sagemaker=true`, two empty GitHub repos, two CloudFormation
stacks, a seed-code clone, a renamed `pipelines/abalone` directory, a patched
`setup.py`, and a generated `codebuild-buildspec.yml` with ten values string-
substituted into it. Notebook 06 of the original is 68 KB of setup for that.

The equivalent here is four YAML files in `.github/workflows/`, and the thing
they run is the same `python -m churn.*` you ran by hand in notebooks 03-06. CI
and local are the same code path, so CI cannot drift from what you tested.

### The four workflows

| Workflow | Trigger | Does |
| --- | --- | --- |
| `churn / ci` | every PR | ruff, pytest, **and** `dvc status` — fails if `dvc.lock` is stale |
| `churn / train` | `train` label on a PR, or manual | `dvc repro` → train → evaluate → comment metrics on the PR |
| `churn / register` | merge to `main` | retrain, enforce the AUC gate, register, set `@candidate` + `@staging` |
| `churn / deploy` | manual | deploy `@staging` or `@production`, gated by a GitHub Environment |

Note what `churn / ci` catches: you edited `features.py` but did not re-run the
pipeline, so the committed `dvc.lock` no longer describes the committed code.
There is no equivalent check in the original workshop because nothing recorded
what the data was.

### Why training is not automatic on every PR

Training costs money. `churn / train` runs only when you add the `train` label,
so opening a PR cannot quietly start billable jobs.

In [ ]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params)
print("region:", settings.region, "| bucket:", settings.bucket)

In [ ]:
REPO_ROOT = PROJECT_ROOT.parent
for path in sorted((REPO_ROOT / ".github").rglob("*.yml")):
    print(path.relative_to(REPO_ROOT))

## 1. The OIDC role

GitHub mints a short-lived OIDC token; AWS exchanges it for temporary
credentials. **No access keys are stored in the repo.**

Two things to create:
1. An IAM OIDC identity provider for `token.actions.githubusercontent.com`
   (once per account).
2. A role whose trust policy accepts only *your* repo.

In [ ]:
GITHUB_ORG = "<your-github-org-or-username>"   # <-- set this
GITHUB_REPO = "mlops-sagemaker-mlflow"         # <-- and this, if you renamed the repo

assert "<" not in GITHUB_ORG, "set GITHUB_ORG above"

import boto3

ACCOUNT_ID = boto3.client("sts", region_name=settings.region).get_caller_identity()["Account"]
print("account:", ACCOUNT_ID)
print("repo   :", f"{GITHUB_ORG}/{GITHUB_REPO}")

In [ ]:
# Create the OIDC provider if the account does not already have one.
iam = boto3.client("iam")
PROVIDER_ARN = f"arn:aws:iam::{ACCOUNT_ID}:oidc-provider/token.actions.githubusercontent.com"

existing = [p["Arn"] for p in iam.list_open_id_connect_providers()["OpenIDConnectProviderList"]]
if PROVIDER_ARN in existing:
    print("OIDC provider already exists")
else:
    iam.create_open_id_connect_provider(
        Url="https://token.actions.githubusercontent.com",
        ClientIDList=["sts.amazonaws.com"],
        ThumbprintList=["6938fd4d98bab03faadb97b34396831e3780aea1"],
    )
    print("created OIDC provider")

In [ ]:
# The `sub` condition is the security boundary: only workflows in this repo can
# assume the role. Narrow it further with `repo:org/repo:ref:refs/heads/main` if
# you want main-only, but note that blocks the PR-label train workflow.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Federated": PROVIDER_ARN},
            "Action": "sts:AssumeRoleWithWebIdentity",
            "Condition": {
                "StringEquals": {"token.actions.githubusercontent.com:aud": "sts.amazonaws.com"},
                "StringLike": {
                    "token.actions.githubusercontent.com:sub": f"repo:{GITHUB_ORG}/{GITHUB_REPO}:*"
                },
            },
        }
    ],
}
print(json.dumps(trust_policy, indent=2))

In [ ]:
# Least-privilege-ish inline policy. Scoped to this project's bucket prefixes and
# to passing only the SageMaker execution role.
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "SageMakerJobsAndRegistry",
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateTrainingJob",
                "sagemaker:DescribeTrainingJob",
                "sagemaker:CreateTransformJob",
                "sagemaker:DescribeTransformJob",
                "sagemaker:CreateModel",
                "sagemaker:DescribeModel",
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:UpdateEndpoint",
                "sagemaker:DescribeEndpoint",
                "sagemaker:DescribeEndpointConfig",
                "sagemaker:InvokeEndpoint",
                "sagemaker:ListModelPackages",
                "sagemaker:DescribeModelPackage",
                "sagemaker:UpdateModelPackage",
                "sagemaker:CreateModelPackage",
                "sagemaker:CreateModelPackageGroup",
                "sagemaker:DescribeModelPackageGroup",
                "sagemaker:ListTags",
                "sagemaker:AddTags",
            ],
            "Resource": "*",
        },
        {
            "Sid": "MLflowTrackingServer",
            "Effect": "Allow",
            "Action": ["sagemaker-mlflow:*", "sagemaker:DescribeMlflowTrackingServer"],
            "Resource": "*",
        },
        {
            "Sid": "ProjectBucket",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{settings.bucket}",
                f"arn:aws:s3:::{settings.bucket}/{settings.bucket_prefix}/*",
                f"arn:aws:s3:::{settings.bucket}/{settings.dvc_remote_prefix}/*",
                f"arn:aws:s3:::{settings.bucket}/mlflow/*",
            ],
        },
        {
            "Sid": "PassExecutionRoleOnly",
            "Effect": "Allow",
            "Action": "iam:PassRole",
            "Resource": settings.sagemaker_role,
            "Condition": {"StringEquals": {"iam:PassedToService": "sagemaker.amazonaws.com"}},
        },
        {
            "Sid": "TrainingLogs",
            "Effect": "Allow",
            "Action": ["logs:GetLogEvents", "logs:DescribeLogStreams", "cloudwatch:GetMetricStatistics"],
            "Resource": "*",
        },
    ],
}
print(json.dumps(permissions_policy, indent=2))

In [ ]:
ROLE_NAME = "churn-mlops-github-actions"

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="OIDC role for the churn-mlops GitHub Actions workflows",
        MaxSessionDuration=7200,
    )["Role"]
    print("created", role["Arn"])
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust_policy))
    role = iam.get_role(RoleName=ROLE_NAME)["Role"]
    print("updated trust policy on", role["Arn"])

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="churn-mlops-access",
    PolicyDocument=json.dumps(permissions_policy),
)
GHA_ROLE_ARN = role["Arn"]
print("role arn:", GHA_ROLE_ARN)

## 2. Repository variables

The workflows read five Actions **variables** (not secrets — none of these is a
credential, and having them visible makes failed runs debuggable).

In [ ]:
variables = {
    "AWS_REGION": settings.region,
    "AWS_GHA_ROLE_ARN": GHA_ROLE_ARN,
    "CHURN_S3_BUCKET": settings.bucket,
    "CHURN_SAGEMAKER_ROLE": settings.sagemaker_role,
    "MLFLOW_TRACKING_URI": settings.mlflow_tracking_uri,
}

print("With the gh CLI:\n")
for key, value in variables.items():
    print(f'gh variable set {key} --body "{value}"')

print("\nOr: Settings -> Secrets and variables -> Actions -> Variables tab.")

In [ ]:
# If the gh CLI is available and authenticated, set them now.
import subprocess

if subprocess.run(["which", "gh"], capture_output=True).returncode == 0:
    for key, value in variables.items():
        result = subprocess.run(
            ["gh", "variable", "set", key, "--body", value],
            capture_output=True, text=True, cwd=REPO_ROOT,
        )
        print(f"{key}: {'ok' if result.returncode == 0 else result.stderr.strip()}")
else:
    print("gh CLI not installed - set the variables in the GitHub UI using the commands above")

## 3. Environments (the approval gate)

Create two environments under **Settings → Environments**:

- **`churn-staging`** — no protection. Deploys to staging should be frictionless.
- **`churn-production`** — add **required reviewers**.

That second one *is* the gate. The deploy job declares
`environment: churn-${{ inputs.environment }}`, so a production deploy pauses
until a reviewer approves, and GitHub records who approved what.

This is the replacement for the workshop's `PendingManualApproval` model-version
status plus the CodePipeline manual-approval stage — same control, one place,
and the audit trail sits next to the code review.

In [ ]:
print("gh api --method PUT repos/:owner/:repo/environments/churn-staging")
print("gh api --method PUT repos/:owner/:repo/environments/churn-production")
print("\nThen add required reviewers to churn-production in the UI")
print("(the reviewers field needs user/team IDs, which the UI resolves for you).")

## 4. Push and watch it run

```sh
cd <repo root>
git add .github churn-mlops
git commit -m "churn-mlops: DVC + MLflow pipeline with GitHub Actions CI/CD"
git push
```

`churn / ci` fires on the push. Then:

```sh
gh workflow run "churn / train"
gh run watch
```

In [ ]:
if subprocess.run(["which", "gh"], capture_output=True).returncode == 0:
    print(subprocess.run(["gh", "workflow", "list"], capture_output=True, text=True, cwd=REPO_ROOT).stdout)
else:
    print("install the gh CLI to drive workflows from here")

In [ ]:
# Kick off a training run on the current main.
print('gh workflow run "churn / train" -f run-name=manual-smoke')
print('gh run watch')
print()
print('# Deploy, after a model has been registered:')
print('gh workflow run "churn / deploy" -f environment=staging')
print('gh workflow run "churn / deploy" -f environment=production -f promote=true')

## 5. The loop you now have

```
edit code / params          ->  PR
                                 ├─ churn / ci       ruff, pytest, dvc.lock freshness
                                 └─ label "train"    ->  churn / train   metrics commented on the PR
merge to main               ->  churn / register     gate, register, @candidate + @staging
manual dispatch             ->  churn / deploy       staging, then production behind a reviewer
```

Every artifact in that loop is traceable: the endpoint serves a model version,
the version carries a `git_commit` and `dvc.*` hashes, and those two restore the
exact training data.

---
## Next

**`99-cleanup.ipynb`** -- remove the billable resources.

### What changed vs. the original workshop
No Service Catalog product, no CodeStar connection to tag, no second GitHub repo,
no CloudFormation stacks, no seed code to patch, no generated buildspec. The
`run_pipeline.py` CLI that CodeBuild invoked is replaced by `python -m churn.*`
entry points that you also run locally — so there is no CI-only code path to
debug blind.